In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [2]:
df_train = pd.read_csv('train.csv', dtype={'sample_id': str})
df_test = pd.read_csv('test.csv', dtype={'sample_id': str})

In [3]:
df_train.head()

,sample_id,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,delay
0,0194048,2010,6,YV,Mesa Airlines Inc.,MEM,"Memphis, TN: Memphis International",258.0,63.0,34.00,2.00,9.57,0.0,17.43,7.0,0.0,3514.0
1,0200607,2010,1,MQ,American Eagle Airlines Inc.,SAN,"San Diego, CA: San Diego International",512.0,61.0,17.47,5.49,16.85,0.0,21.19,15.0,0.0,2444.0
2,0253640,2007,3,OO,SkyWest Airlines Inc.,BTV,"Burlington, VT: Burlington International",66.0,27.0,11.00,0.00,0.00,0.0,16.00,3.0,0.0,1421.0
3,0061936,2019,4,EV,ExpressJet Airlines LLC,PVD,"Providence, RI: Theodore Francis Green State",89.0,29.0,9.84,0.00,3.74,0.0,15.42,5.0,0.0,2376.0
4,0220886,2008,11,DL,Delta Air Lines Inc.,SDF,"Louisville, KY: Louisville Muhammad Ali Intern...",82.0,24.0,7.79,0.77,6.08,0.0,9.36,0.0,1.0,1505.0


In [4]:
df_train.isna().sum()

sample_id             0
year                  0
month                 0
carrier               4
carrier_name          4
airport               2
airport_name          0
arr_flights         397
arr_del15           576
carrier_ct          394
weather_ct          395
nas_ct              392
security_ct         392
late_aircraft_ct    392
arr_cancelled       392
arr_diverted        393
delay               397
dtype: int64

In [5]:
df_train = df_train[df_train['delay'].notna()].reset_index(drop=True)

# delay = arr_del15 * (minute medii per zbor intarziat)
(df_train['delay'] / df_train['arr_del15']).describe()

count    246697.000000
mean         55.132741
std          26.186015
min          15.000000
25%          41.738462
50%          51.489691
75%          63.191083
max        1551.000000
dtype: float64

In [6]:
ct_list = ['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct']
cat_list = ['carrier', 'airport', 'carrier_airport', 'airport_year', 'carrier_year', 'ym', 'airport_ym', 'carrier_ym']

def add_features(df):
    df['carrier'] = df['carrier'].fillna('NA')
    df['airport'] = df['airport'].fillna('NA')
    df['arr_del15'] = df['arr_del15'].fillna(df[ct_list].sum(axis=1).round())

    del15 = df['arr_del15'].clip(lower=1)
    for col in ct_list:
        df[col + '_frac'] = df[col] / del15

    flights = df['arr_flights'].clip(lower=1)
    df['del_rate'] = df['arr_del15'] / flights
    df['cancel_rate'] = df['arr_cancelled'] / flights

    df['ym'] = df['year'].astype(str) + '_' + df['month'].astype(str)
    df['carrier_airport'] = df['carrier'] + '_' + df['airport']
    df['airport_year'] = df['airport'] + '_' + df['year'].astype(str)
    df['carrier_year'] = df['carrier'] + '_' + df['year'].astype(str)
    df['airport_ym'] = df['airport'] + '_' + df['ym']
    df['carrier_ym'] = df['carrier'] + '_' + df['ym']

def process(df_train, df_test):
    add_features(df_train)
    add_features(df_test)

    df_train.drop(['carrier_name', 'airport_name'], axis=1, inplace=True)
    df_test.drop(['carrier_name', 'airport_name'], axis=1, inplace=True)

process(df_train, df_test)

C:\Users\david\AppData\Local\Temp\ipykernel_37188\2507184518.py:28: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_train.drop(['carrier_name', 'airport_name'], axis=1, inplace=True)
C:\Users\david\AppData\Local\Temp\ipykernel_37188\2507184518.py:29: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_test.drop(['carrier_name', 'airport_name'], axis=1, inplace=True)


In [7]:
df_train.columns

Index(['sample_id', 'year', 'month', 'carrier', 'airport', 'arr_flights',
       'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct',
       'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'delay',
       'carrier_ct_frac', 'weather_ct_frac', 'nas_ct_frac', 'security_ct_frac',
       'late_aircraft_ct_frac', 'del_rate', 'cancel_rate', 'ym',
       'carrier_airport', 'airport_year', 'carrier_year', 'airport_ym',
       'carrier_ym'],
      dtype='str')

In [8]:
df_fit = df_train[df_train['arr_del15'] > 0]

X = df_fit.drop(['sample_id', 'delay'], axis=1)
y = df_fit['delay'] / df_fit['arr_del15']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
def evaluate_model(model):
    y_train_pred = model.predict(X_train) * X_train['arr_del15']
    y_test_pred = model.predict(X_test) * X_test['arr_del15']

    print(f"Train: {mean_absolute_error(y_train * X_train['arr_del15'], y_train_pred)}, Test: {mean_absolute_error(y_test * X_test['arr_del15'], y_test_pred)}")

In [10]:
from catboost import CatBoostRegressor

In [11]:
cbr = CatBoostRegressor(iterations=3000, depth=8, learning_rate=0.08, cat_features=cat_list, verbose=500)
cbr.fit(X_train, y_train, sample_weight=X_train['arr_del15'])
evaluate_model(cbr)

0:	learn: 14.5630479	total: 282ms	remaining: 14m 4s


500:	learn: 8.2479813	total: 1m 38s	remaining: 8m 12s


1000:	learn: 7.8301563	total: 3m 24s	remaining: 6m 48s


1500:	learn: 7.5909956	total: 5m 9s	remaining: 5m 8s


2000:	learn: 7.4140726	total: 6m 54s	remaining: 3m 27s


2500:	learn: 7.2582276	total: 8m 47s	remaining: 1m 45s


2999:	learn: 7.1244904	total: 10m 56s	remaining: 0us


Train: 313.5576812943527, Test: 393.27620807973057


In [12]:
cbr_full = CatBoostRegressor(iterations=3000, depth=8, learning_rate=0.08, cat_features=cat_list, verbose=500)
cbr_full.fit(X, y, sample_weight=X['arr_del15'])

0:	learn: 14.5951376	total: 314ms	remaining: 15m 40s


500:	learn: 8.3891972	total: 2m 7s	remaining: 10m 35s


1000:	learn: 7.9444113	total: 4m 10s	remaining: 8m 20s


1500:	learn: 7.6797227	total: 6m 20s	remaining: 6m 19s


2000:	learn: 7.4855117	total: 8m 27s	remaining: 4m 13s


2500:	learn: 7.3418502	total: 10m 28s	remaining: 2m 5s


2999:	learn: 7.2100528	total: 12m 22s	remaining: 0us


CatBoostRegressor(cat_features=['carrier', 'airport', 'carrier_airport', 'airport_year', 'carrier_year', 'ym', 'airport_ym', 'carrier_ym'], depth=8, iterations=3000, learning_rate=0.08, loss_function='RMSE', verbose=500)

In [13]:
def solve(model):
    X_pred = df_test.drop(['sample_id'], axis=1)
    y_pred = model.predict(X_pred) * df_test['arr_del15']
    y_pred = np.where(df_test['arr_del15'] > 0, y_pred, 0)

    output = pd.DataFrame({'sample_id': df_test['sample_id'], 'delay': np.round(np.clip(y_pred, 0, None)).astype(int)})
    output.to_csv('submission.csv', index=False)
solve(cbr_full)